In [ ]:
# !pip install --upgrade optree

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler

In [3]:
# Reproducibility
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed = 1
set_seed(seed)

# Load outputs
X = np.load("/content/X_ts.npy").astype(np.float32)
meta = pd.read_csv("/content/meta.csv")
ctx = pd.read_csv("/content/X_ctx.csv")

print("X shape:", X.shape)
print(meta.head())

# Labels produced by your preprocessing script are already zero-indexed
y = meta["y"].to_numpy(dtype=np.int64)

# Optional sanity checks
assert len(X) == len(meta), "Mismatch between X_ts.npy and meta.csv"
assert X.ndim == 3, "Expected X shape to be (N, C, T)"

X shape: (2500, 7, 256)
              BDD_ID  EVENT_ID  EVENT_TYPE  y  event_time_ms anchor_source  \
0  03d767c8-2eca52c7         1           2  1   1.504000e+12     BDD_START   
1  02ef72c1-950d6851         2           2  1   1.503718e+12     BDD_START   
2  04200e90-4a4c631e         3           2  1   1.507228e+12     BDD_START   
3  04a8e132-4f929465         4           1  0   1.504306e+12     BDD_START   
4  0549f27d-2e4ab1af         5           2  1   1.507457e+12     BDD_START   

                                      info_json_path  label_ts_ms  
0  data/bddk100_info/100k/train/03d767c8-2eca52c7...        10000  
1  data/bddk100_info/100k/train/02ef72c1-950d6851...        10000  
2  data/bddk100_info/100k/train/04200e90-4a4c631e...        10000  
3  data/bddk100_info/100k/train/04a8e132-4f929465...        10000  
4  data/bddk100_info/100k/train/0549f27d-2e4ab1af...        10000  


In [6]:
# Split by BDD_ID so the same ride/clip does not leak
# across train/val/test
groups = meta["BDD_ID"].astype(str).to_numpy()

# First split: train (70%) vs temp (30%)
gss1 = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=seed)
train_idx, temp_idx = next(gss1.split(X, y, groups=groups))

X_train, y_train = X[train_idx], y[train_idx]
X_temp, y_temp = X[temp_idx], y[temp_idx]
groups_temp = groups[temp_idx]

# Second split: val (15%) vs test (15%) from temp
gss2 = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=seed)
val_idx_rel, test_idx_rel = next(gss2.split(X_temp, y_temp, groups=groups_temp))

X_val, y_val = X_temp[val_idx_rel], y_temp[val_idx_rel]
X_test, y_test = X_temp[test_idx_rel], y_temp[test_idx_rel]

print("Train:", X_train.shape, y_train.shape)
print("Val:  ", X_val.shape, y_val.shape)
print("Test: ", X_test.shape, y_test.shape)

Train: (1750, 7, 256) (1750,)
Val:   (375, 7, 256) (375,)
Test:  (375, 7, 256) (375,)


In [7]:
# Separate features from labels/IDs in ctx
ctx_features = ctx.drop(columns=["BDD_ID", "EVENT_ID", "EVENT_TYPE", "y"])

# One-hot encode categorical variables
categorical_cols = ["weather", "scene", "timeofday"]
ctx_features = pd.get_dummies(ctx_features, columns=categorical_cols, dummy_na=True)

# Fill NaNs (using 0 for counts/areas makes sense physically here)
ctx_features = ctx_features.fillna(0)

# Convert to float32 numpy array
X_ctx_np = ctx_features.to_numpy(dtype=np.float32)

# split the context data using the EXACT SAME INDICES
# First split
X_ctx_train, X_ctx_temp = X_ctx_np[train_idx], X_ctx_np[temp_idx]

# Second split
X_ctx_val, X_ctx_test = X_ctx_temp[val_idx_rel], X_ctx_temp[test_idx_rel]

# Scale the numerical context features (FIT ONLY ON TRAIN to prevent data leakage)
scaler = StandardScaler()
X_ctx_train = scaler.fit_transform(X_ctx_train)
X_ctx_val = scaler.transform(X_ctx_val)
X_ctx_test = scaler.transform(X_ctx_test)

in_chans_ctx = X_ctx_train.shape[1]
print("Context feature size:", in_chans_ctx)

Context feature size: 37


### I've revised the class from Arvin's Paper

In [14]:
class MultiModal_CNN_LSTM(nn.Module):
    def __init__(self, in_chans_ts, seq_len, in_chans_ctx, num_classes):
        super(MultiModal_CNN_LSTM, self).__init__()

        # BRANCH 1: Time-Series (Kinematics)
        self.bn_input = nn.BatchNorm1d(in_chans_ts)
        self.conv1 = nn.Conv1d(in_channels=in_chans_ts, out_channels=64, kernel_size=12, stride=1, padding='same')
        self.bn_conv = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.maxpool1 = nn.MaxPool1d(kernel_size=2, stride=2)
        self.lstm = nn.LSTM(input_size=64, hidden_size=100, num_layers=2, batch_first=True, dropout=0.3)

        # BRANCH 2: Tabular Context (Weather, Objects, etc.)
        self.ctx_fc1 = nn.Linear(in_chans_ctx, 64)
        self.ctx_relu = nn.ReLU()
        self.ctx_bn = nn.BatchNorm1d(64)
        self.ctx_drop = nn.Dropout(0.3)

        # FUSION: Combine Branch 1 (100 features) and Branch 2 (64 features)
        fusion_size = 100 + 64

        # FINAL CLASSIFIER
        self.fusion_drop = nn.Dropout(0.5)
        self.fc_final = nn.Linear(fusion_size, num_classes)

    def forward(self, x_ts, x_ctx):
        # Process Time-Series
        x_ts = self.bn_input(x_ts)
        x_ts = self.conv1(x_ts)
        x_ts = self.bn_conv(x_ts)
        x_ts = self.relu1(x_ts)
        x_ts = self.maxpool1(x_ts)
        lstm_out, _ = self.lstm(x_ts.permute(0, 2, 1))
        x_ts_features, _ = torch.max(lstm_out, dim=1)  # Global Max Pooling -> Output is size 100

        # Process Context
        x_ctx_features = self.ctx_fc1(x_ctx)
        x_ctx_features = self.ctx_bn(x_ctx_features)
        x_ctx_features = self.ctx_relu(x_ctx_features)
        x_ctx_features = self.ctx_drop(x_ctx_features) # Output is size 64

        # FUSION (Concatenate along feature dimension)
        fused = torch.cat((x_ts_features, x_ctx_features), dim=1) # Output is size 164

        # Final Classification
        out = self.fusion_drop(fused)
        out = self.fc_final(out)
        return out

In [15]:
class MultiModalDataset(Dataset):
    def __init__(self, X_ts, X_ctx, y):
        self.X_ts = X_ts
        self.X_ctx = X_ctx
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        X_ts_sample = self.X_ts[idx]
        X_ctx_sample = self.X_ctx[idx]
        y_sample = torch.tensor(self.y[idx], dtype=torch.int64).item()
        return X_ts_sample, X_ctx_sample, y_sample

# Re-initialize DataLoaders with the new dataset
train_dataset = MultiModalDataset(X_train, X_ctx_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)

val_dataset = MultiModalDataset(X_val, X_ctx_val, y_val)
val_loader = DataLoader(val_dataset, batch_size=300, shuffle=False, num_workers=4)

test_dataset = MultiModalDataset(X_test, X_ctx_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=300, shuffle=False, num_workers=4)

In [18]:
in_chans_ts = X_train.shape[1]
seq_len = X_train.shape[2]
num_classes = len(np.unique(y_train))

num_epochs = 220
print_every = 10

# Initialize new multimodal model
model = MultiModal_CNN_LSTM(
    in_chans_ts=in_chans_ts,
    seq_len=seq_len,
    in_chans_ctx=in_chans_ctx,
    num_classes=num_classes
)
model.to(device)

class_counts = np.bincount(y_train, minlength=num_classes)
class_weights = len(y_train) / (len(class_counts) * class_counts)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Update evaluation function to unpack both inputs
def evaluate_test_loss(model, test_loader, criterion):
    model.eval()
    running_test_loss = 0.0
    with torch.no_grad():
        for inputs_ts, inputs_ctx, labels in test_loader:
            inputs_ts = inputs_ts.to(device)
            inputs_ctx = inputs_ctx.to(device)
            labels = labels.to(device)

            outputs = model(inputs_ts, inputs_ctx)
            test_loss = criterion(outputs, labels)

            running_test_loss += test_loss.item()
    return running_test_loss / len(test_loader)

# Early stopping
best_val_loss = float('inf')
patience = 25 # How many epochs to wait before giving up
patience_counter = 0

# Update training loop to unpack both inputs
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (inputs_ts, inputs_ctx, labels) in enumerate(train_loader, 0):
        inputs_ts = inputs_ts.to(device)
        inputs_ctx = inputs_ctx.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs_ts, inputs_ctx)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    val_loss = evaluate_test_loss(model, val_loader, criterion)
    train_loss = running_loss / len(train_loader)

    # saves best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model.pth")
        patience_counter = 0 # reset patience
    else:
        patience_counter += 1

    if (epoch + 1) % print_every == 0:
        print(f"Epoch: {epoch + 1}/{num_epochs}, Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    # Stop training early if val_loss hasn't improved in 25 epochs
    if patience_counter >= patience:
        print(f"Early stopping triggered at epoch {epoch + 1}!")
        break

print("Training completed!")

# load best model for evaluation
model.load_state_dict(torch.load("best_model.pth"))

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/conv.py:370: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv1d(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch: 10/220, Train Loss: 0.9538, Val Loss: 1.0691
Epoch: 20/220, Train Loss: 0.8044, Val Loss: 1.0012
Epoch: 30/220, Train Loss: 0.7508, Val Loss: 0.9775
Epoch: 40/220, Train Loss: 0.6964, Val Loss: 0.9779
Epoch: 50/220, Train Loss: 0.6238, Val Loss: 1.0400
Epoch: 60/220, Train Loss: 0.5820, Val Loss: 1.0287
Early stopping triggered at epoch 63!
Training completed!


<All keys matched successfully>

In [19]:
# Set the model to evaluation mode
model.eval()

# Initialize variables to store true and predicted labels
y_true = []
y_pred = []
y_prob = []

# Update test data evaluation to unpack both inputs
with torch.no_grad():
    for inputs_ts, inputs_ctx, labels in test_loader:
        inputs_ts = inputs_ts.to(device)
        inputs_ctx = inputs_ctx.to(device)
        labels = labels.to(device)

        outputs = model(inputs_ts, inputs_ctx)
        predicted = outputs.argmax(dim=1)
        probabilities = F.softmax(outputs, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())
        y_prob.extend(probabilities.cpu().numpy())

# Calculate evaluation metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average=None)
recall = recall_score(y_true, y_pred, average=None)
f1 = f1_score(y_true, y_pred, average=None)
confusion_mat = confusion_matrix(y_true, y_pred)
auc = roc_auc_score(y_true, y_prob, multi_class='ovr')

# Print evaluation metrics
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)
print("AUC Score:", auc)
print("Confusion Matrix:\n", confusion_mat)

Accuracy: 0.5893333333333334
Precision: [0.46236559 0.45714286 0.49038462 0.77622378]
Recall: [0.41747573 0.53333333 0.6375     0.68518519]
F1 Score: [0.43877551 0.49230769 0.55434783 0.72786885]
AUC Score: 0.8059165199002334
Confusion Matrix:
 [[ 43   4  35  21]
 [  6  16   2   6]
 [ 22   2  51   5]
 [ 22  13  16 111]]


In [ ]:
len(y_train)

1750